# BERT for Quora Question Pairs

In [1]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from datasets import load_dataset
import lightning as L

In [2]:
DATASET_NAME = "AlekseyKorshuk/quora-question-pairs"
MODEL_NAME = "google-bert/bert-base-uncased"
SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 4
D_MODEL = 768
LEARNING_RATE = 1e-3
MAX_EPOCHS = 3

## 1. Prepare dataset

In [3]:
ds = load_dataset(DATASET_NAME, split="train")

In [4]:
ds

Dataset({
    features: ['id', 'qid1', 'qid2', 'question1', 'question2', 'is_duplicate'],
    num_rows: 404290
})

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
ds_cleaned = ds.filter(
    lambda row: isinstance(row["question1"], str) and isinstance(row["question2"], str)
)
ds_tokenized = ds_cleaned.map(
    lambda row: tokenizer(row["question1"], row["question2"]),
    batched=True,
    remove_columns=["question1", "question2"],
)

In [7]:
ds_split_1 = ds_tokenized.train_test_split(0.3)
ds_split_2 = ds_split_1["test"].train_test_split(0.5)
ds_train, ds_val, ds_test = ds_split_1["train"], ds_split_2["train"], ds_split_2["test"]

In [8]:
ds_train, ds_val, ds_test

(Dataset({
     features: ['id', 'qid1', 'qid2', 'is_duplicate', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 283000
 }),
 Dataset({
     features: ['id', 'qid1', 'qid2', 'is_duplicate', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 60643
 }),
 Dataset({
     features: ['id', 'qid1', 'qid2', 'is_duplicate', 'input_ids', 'token_type_ids', 'attention_mask'],
     num_rows: 60644
 }))

In [9]:
data_collator = DataCollatorWithPadding(tokenizer)

In [10]:
train_loader = DataLoader(
    ds_train,
    BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
val_loader = DataLoader(
    ds_val,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)
test_loader = DataLoader(
    ds_test,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=data_collator,
)

In [11]:
for batch in test_loader:
    yo = batch
    break

In [12]:
yo

{'id': tensor([168883, 315132, 387714, 192863, 224391, 132956, 129122, 260493,  41013,
        119803, 400086, 266608, 361900, 201901, 146288, 324067, 321381, 392860,
        381865,    516, 132498, 297510, 308151, 141370, 318241, 138892, 328352,
        132731, 310980, 401623, 340229, 313157]), 'qid1': tensor([ 30901, 439985, 520051, 292607, 332462, 212781, 207504,  56865,  74120,
        194438,   7401, 383808, 320184, 304026, 231121, 450116, 447073, 314780,
        322203,   1030, 212146,  71817, 431950, 224448, 443558, 143452, 454870,
        212474,  45323,  46045, 318799, 231985]), 'qid2': tensor([ 74045, 439986, 376152, 292608, 332463, 212782, 207505,  59223,  74121,
        194439,  22536, 383809,  83623,  54189,   9146, 450117, 177990, 280134,
        341695,   1031, 212147, 419883, 431951, 224449, 443559, 221073, 454871,
        212475,  31619,   3624, 434851,  28764]), 'is_duplicate': tensor([1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0,
        0, 

## 2. Prepare model

In [42]:
class BertQuoraModel(L.LightningModule):
    def __init__(self, bert, d_model = D_MODEL):
        super().__init__()
        self.bert = bert
        self.d_model = d_model
        self.classifier = nn.Linear(d_model, 1)
    
    def forward(self, input_ids, attention_mask, token_type_ids):
        bert_output = self.bert(input_ids, attention_mask, token_type_ids)
        cls_token = bert_output.pooler_output
        logits = self.classifier(cls_token).reshape(-1)
        return logits
    
    def training_step(self, batch, batch_idx):
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        labels = batch["is_duplicate"].float()

        logits = self(input_ids, attention_mask, token_type_ids)
        loss = F.binary_cross_entropy_with_logits(logits, labels)
        
        preds = logits > 0
        accuracy = (preds == labels).float().mean()
        tp = preds.logical_and(labels).sum()
        fp = preds.logical_and(labels.logical_not()).sum()
        fn = preds.logical_not().logical_and(labels).sum()
        precision = tp / (tp + fp)
        recall = tp / (tp + fn)
        f1 = 2 * precision * recall / (precision + recall)

        self.log("train_loss", loss)
        self.log("train_accuracy", accuracy)
        self.log("train_precision", precision)
        self.log("train_recall", recall)
        self.log("train_f1", f1)

        return loss
    
    def on_train_start(self):
        self.train()
        
    
    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), LEARNING_RATE)
        return optimizer

In [43]:
bert = AutoModel.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
bert

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

## 3. Train model

In [45]:
model = BertQuoraModel(bert)

In [ ]:
trainer = L.Trainer(max_epochs=1, limit_train_batches=100)
trainer.fit(model, train_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ bert       │ BertModel │  109 M │ eval  │     0 │
│ 1 │ classifier │ Linear    │    769 │ train │     0 │
└───┴────────────┴───────────┴────────┴───────┴───────┘

Trainable params: 109 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 109 M                                                                                                
Total estimated model params size (MB): 437.932                                                                    
Modules in train mode: 1                                                                                           
Modules in eval mode: 228                                                                                          
Total FLOPs: 0

Output()

In [ ]:
yo["is_duplicate"].dtype

torch.int64